In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
import gc
import subprocess
import os
import time
from openai import OpenAI
import time
import pandas as pd
from tqdm import tqdm
import time
import pandas as pd
from tqdm import tqdm


In [2]:
unsup_lora_list = [f"../{k}" for k in os.listdir("../") if "lora" in k and "unsup" in k]
unsup_lora_list 

['../unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_3_r_32_alpha_64_lr_0.0001_final_2_lora',
 '../unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_5_r_16_alpha_32_lr_0.0002_final_1_lora',
 '../unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_3_r_16_alpha_32_lr_0.0002_final_3_lora',
 '../unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_5_r_16_alpha_32_lr_5e-05_final_4_lora',
 '../unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_3_r_16_alpha_32_lr_0.0002_final_0_lora']

In [3]:
qa_lora_list = [f"../{k}" for k in os.listdir("../") if "lora" in k and "unsup" not in k]
qa_lora_list 

['../meta-llama_Llama-3.1-8B-Instruct_lora_epochs_5_batch_1_optim_paged_adamw_8bit_r_16_alpha_32_lr_5e-06_drop_out_0.05_q_proj_v_proj_k_proj_o_proj',
 '../meta-llama_Llama-3.1-8B-Instruct_lora_epochs_3_batch_1_optim_paged_adamw_8bit_r_32_alpha_64_lr_1e-05_drop_out_0.05_q_proj_v_proj_k_proj_o_proj',
 '../meta-llama_Llama-3.1-8B-Instruct_lora_epochs_3_batch_1_optim_paged_adamw_8bit_r_16_alpha_32_lr_1e-05_drop_out_0.05_q_proj_v_proj_k_proj_o_proj',
 '../meta-llama_Llama-3.1-8B-Instruct_lora_epochs_3_batch_1_optim_paged_adamw_8bit_r_16_alpha_32_lr_1e-05_drop_out_0.05_q_proj_k_proj_v_proj_o_proj_gate_proj_up_proj_down_proj',
 '../meta-llama_Llama-3.1-8B-Instruct_lora_epochs_3_batch_1_optim_paged_adamw_8bit_r_16_alpha_32_lr_5e-06_drop_out_0.05_q_proj_v_proj_k_proj_o_proj']

In [4]:
unsup_fft_list = [f"../{k}" for k in os.listdir("../") if "fft" in k and "unsup" in k]
unsup_fft_list 

['../FFT_UNSUP_meta-llama_Llama-3.1-8B-Instruct_dataset_txt_epochs_5_maxlen_512_bs_1_ga_8_lr_5e-06_unsup_fft_2',
 '../FFT_UNSUP_meta-llama_Llama-3.1-8B-Instruct_dataset_txt_epochs_3_maxlen_1024_bs_1_ga_12_lr_5e-06_save_unsup_fft_4',
 '../FFT_UNSUP_meta-llama_Llama-3.1-8B-Instruct_dataset_txt_epochs_3_maxlen_1024_bs_1_ga_12_lr_1e-05_save_unsup_fft_3',
 '../FFT_UNSUP_meta-llama_Llama-3.1-8B-Instruct_dataset_txt_epochs_3_maxlen_512_bs_1_ga_8_lr_1e-05_save_unsup_fft_0',
 '../FFT_UNSUP_meta-llama_Llama-3.1-8B-Instruct_dataset_txt_epochs_3_maxlen_512_bs_1_ga_8_lr_5e-06_save_unsup_fft_1']

In [5]:
qa_fft_list = [f"../{k}" for k in os.listdir("../") if "fft" in k and "qa" in k]
qa_fft_list

['../meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_3_maxlen_512_bs_1_collator_old_save_fft_qa_1',
 '../meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_5_maxlen_1024_bs_1_collator_new_save_fft_qa_4',
 '../meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_3_maxlen_1024_bs_1_collator_new_save_fft_qa_3',
 '../meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_3_maxlen_512_bs_1_collator_new_save_fft_qa_0',
 '../meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_5_maxlen_512_bs_1_collator_new_save_fft_qa_2']

In [6]:
def api_ai_answer(client, prompt):

    response = client.chat.completions.create(
        model="LORA_THE_BEST",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000,
        temperature=0.0
    )
    
    # Извлекаем текст ответа
    answer = response.choices[0].message.content
 
    return answer

In [7]:
from concurrent.futures import ThreadPoolExecutor

In [ ]:
def test_model_adapter(base_model_name, adapter_path=None, with_adapter=True, t_mode = False):
    os.makedirs("DUMPS_VLLM", exist_ok=True)
    os.makedirs("OUTPUTS", exist_ok=True)

    if with_adapter:
        safe_name = adapter_path.replace("/", "_").replace("..", "")

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        ).to("cuda")

        tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
        tokenizer.pad_token = tokenizer.eos_token

        model = PeftModel.from_pretrained(base_model, adapter_path).to("cuda")
        merged_model = model.merge_and_unload()

        output_dir = "LORA_THE_BEST"
        merged_model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

        del merged_model
        del model
        del base_model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()

        log_path = f"DUMPS_VLLM/vllm_server_{safe_name}.log"

        command = [
            "vllm", "serve",
            "LORA_THE_BEST",
            "--host", "0.0.0.0",
            "--port", "8001",
            "--api-key", "",
            "--served-model-name", "LORA_THE_BEST",
        
            "--dtype", "bfloat16",
        
            "--max-model-len", "4096",
            "--max-num-seqs", "64",
            "--max-num-batched-tokens", "4092",
            "--gpu-memory-utilization", "0.90",
        
            "--enable-prefix-caching",
        ]

    else:
        safe_name = base_model_name.replace("/", "_")

        llama3_chat_template = (
            "{% set loop_messages = messages %}"
            "{% for message in loop_messages %}"
            "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}"
            "{% if loop.index0 == 0 %}"
            "{% set content = bos_token + content %}"
            "{% endif %}"
            "{{ content }}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
            "{% endif %}"
        )

        log_path = f"DUMPS_VLLM/vllm_server_{safe_name}.log"

        command = [
            "vllm", "serve",
            base_model_name,
            "--host", "0.0.0.0",
            "--port", "8001",
            "--api-key", "",
            "--served-model-name", "LORA_THE_BEST",
            "--dtype", "bfloat16",
            "--max-model-len", "4096",
            "--max-num-seqs", "64",
            "--chat-template", llama3_chat_template,
            "--gpu-memory-utilization", "0.90",
        
            "--enable-prefix-caching",
        ]

    log_file = open(log_path, "w")

    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True
    )

    ready = False
    while not ready:
        time.sleep(1)
        with open(log_path, "r") as f:
            if "Application startup complete" in f.read():
                ready = True
                break

    client = OpenAI(
        base_url="http://localhost:8001/v1",
        api_key=""
    )

    df = pd.read_excel("Q_F_T.xlsx")
    if t_mode:
        df = df[:5]

    """
    outputs = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Обработка запросов"):
        prompt = row["Promts"]
        answer = api_ai_answer(client, prompt)

        outputs.append({
            "answer": answer,
        })

    answers = [o["answer"] for o in outputs]
    df["Answers"] = answers
    """

    def call_one_prompt(prompt):
        return api_ai_answer(client, prompt)
    
    prompts = df["Promts"].tolist()
    
    with ThreadPoolExecutor(max_workers=16) as executor:
        answers = list(
            tqdm(
                executor.map(call_one_prompt, prompts),
                total=len(prompts),
                desc="Обработка запросов"
            )
        )
    
    df["Answers"] = answers

    ft_type = ""
    corpus_type = ""
    if with_adapter:
        ft_type = "lora"
        df["ft_type"] = ft_type

        if "unsup" in adapter_path:
            corpus_type = "unsup"
        else:
            corpus_type = "qa"
            
        df["corpus_type"] = corpus_type
        
        df.to_excel(f"OUTPUTS/VLLM_TEST_lora_{safe_name}.xlsx", index=False)
        
    else:
        ft_type = "fft"
        df["ft_type"] = ft_type

        if "unsup" in base_model_name:
            corpus_type = "unsup"
        elif "qa" in base_model_name:
            corpus_type = "qa"
        df["corpus_type"] = corpus_type
        
        df.to_excel(f"OUTPUTS/VLLM_TEST_fft_model_{safe_name}.xlsx", index=False)

    process.terminate()
    log_file.close()

In [9]:
test_model_adapter(unsup_fft_list[0], with_adapter = False, t_mode = True)

Обработка запросов: 100%|██████████| 5/5 [00:05<00:00,  1.06s/it]


In [10]:
test_model_adapter(qa_fft_list[0], with_adapter = False, t_mode = True)

Обработка запросов: 100%|██████████| 5/5 [00:05<00:00,  1.09s/it]


In [11]:
test_model_adapter("meta-llama/Llama-3.1-8B-Instruct", adapter_path=qa_lora_list[0],with_adapter = True, t_mode = True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Обработка запросов: 100%|██████████| 5/5 [00:06<00:00,  1.32s/it]


In [12]:
test_model_adapter("meta-llama/Llama-3.1-8B-Instruct", adapter_path=unsup_lora_list[0],with_adapter = True, t_mode = True)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Обработка запросов: 100%|██████████| 5/5 [00:11<00:00,  2.26s/it]


In [13]:
for k in tqdm(unsup_lora_list, desc="Загрузка"):
    test_model_adapter("meta-llama/Llama-3.1-8B-Instruct", adapter_path=k,with_adapter = True, t_mode = False)

Загрузка:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Загрузка:  20%|██        | 1/5 [02:15<09:01, 135.46s/it]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Загрузка:  80%|████████  | 4/5 [08:57<02:15, 135.49s/it]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Загрузка: 100%|██████████| 5/5 [11:13<00:00, 134.80s/it]


In [14]:
for k in tqdm(qa_lora_list, desc="Загрузка"):
    test_model_adapter("meta-llama/Llama-3.1-8B-Instruct", adapter_path=k,with_adapter = True, t_mode = False)

Загрузка:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Загрузка:  20%|██        | 1/5 [01:38<06:32, 98.16s/it]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Загрузка:  80%|████████  | 4/5 [06:43<01:41, 101.32s/it]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Загрузка: 100%|██████████| 5/5 [08:22<00:00, 100.52s/it]


In [15]:
for k in tqdm(unsup_fft_list, desc="Загрузка"):
    test_model_adapter(k, with_adapter = False, t_mode = False)

Загрузка: 100%|██████████| 5/5 [03:20<00:00, 40.16s/it]


In [16]:
for k in tqdm(qa_fft_list, desc="Загрузка"):
    test_model_adapter(k, with_adapter = False, t_mode = False)

Загрузка: 100%|██████████| 5/5 [04:20<00:00, 52.05s/it]
